In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from agentic_rag import (ask, check_input, check_output, clear_cache, override_settings, show_guardrail_check,
                         show_result, show_settings, show_trace_table, show_users)

show_settings()
show_users()
clear_cache()  # start every run from an empty semantic cache, so the scenarios are repeatable

Agent settings (.env)                                                                                              
┌─────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────┐
│ Orchestrator model                      │ gpt-5.5                                                               │
│ Reranker / guardrail model              │ gpt-5.1                                                               │
│ Judge model (RAGAS)                     │ gpt-6.1-sol                                                           │
│ Embedding model                         │ text-embedding-3-small                                                │
│ Chroma                                  │ localhost:8000 {'networking': 'networking_kb', 'meetings':            │
│                                         │ 'meetings_kb'}                                                        │
│ Redis cache                             │ redis://localhost:6379                                                │
│ Cache similarity                        │ 0.9                                                                   │
│ Relevance threshold                     │ 0.6                                                                   │
│ Faithfulness / groundedness / precision │ 0.7 / 0.7 / 0.6                                                       │
│ Context token budget                    │ 6000                                                                  │
│ Max rewrites / retries                  │ 3 / 3                                                                 │
│ Timeout budget (s)                      │ 90.0                                                                  │
│ Max plan steps                          │ 4                                                                     │
└─────────────────────────────────────────┴───────────────────────────────────────────────────────────────────────┘

Demo users (RBAC)                                          
┏━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ User  ┃ Name  ┃ Role     ┃ Allowed access levels        ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ alice │ Alice │ guest    │ public                       │
│ bob   │ Bob   │ employee │ public, internal             │
│ carol │ Carol │ admin    │ public, internal, restricted │
└───────┴───────┴──────────┴──────────────────────────────┘

In [2]:
with override_settings(timeout_seconds=300):
    show_result(ask("Our NetCore Meet calls show MTG-MED-4102 packet loss at branch sites. Which SD-WAN QoS settings "
                    "and SLA class should we apply, and what does the Meet troubleshooting guide recommend?",
                    "bob", verbose=False, mode="plan"), show_trace=False)

Plan v1: The question requires NetCore-specific guidance from both Meet troubleshooting documentation and SD-WAN   
QoS/SLA documentation. Two targeted KB searches will retrieve the Meet error recommendations and the matching      
SD-WAN policy settings for branch packet-loss mitigation.                                                          
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Step ┃ Goal                      ┃ Tool                  ┃ Domains    ┃ Query                      ┃ Depends on ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 1    │ Find the NetCore Meet     │ search_knowledge_base │ meetings   │ NetCore Meet MTG-MED-4102  │ -          │
│      │ troubleshooting guidance  │                       │            │ packet loss                │            │
│      │ for MTG-MED-4102 packet   │                       │            │ troubleshooting guide      │            │
│      │ loss on calls, including  │                       │            │ recommendations branch     │            │
│      │ recommended remediation   │                       │            │ sites calls                │            │
│      │ steps.                    │                       │            │                            │            │
│ 2    │ Find the recommended      │ search_knowledge_base │ networking │ NetCore SD-WAN QoS         │ -          │
│      │ NetCore SD-WAN QoS        │                       │            │ settings SLA class NetCore │            │
│      │ settings and SLA class    │                       │            │ Meet voice video traffic   │            │
│      │ for NetCore Meet voice or │                       │            │ packet loss branch sites   │            │
│      │ video traffic             │                       │            │                            │            │
│      │ experiencing packet loss  │                       │            │                            │            │
│      │ at branch sites.          │                       │            │                            │            │
└──────┴───────────────────────────┴───────────────────────┴────────────┴────────────────────────────┴────────────┘

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Our NetCore Meet calls show MTG-MED-4102 packet loss at branch sites. Which SD-WAN QoS settings and SLA class   │
│ should we apply, and what does the Meet troubleshooting guide recommend?                                        │
│ user=bob (role employee)  mode=plan  status=answered  routes=['vector', 'vector']  rewrites/re-plans=0          │
│ time=154.1s                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ For MTG-MED-4102 at branch sites, treat this as a media-path quality issue and prioritize UDP/9000 stability,   │
│ QoS preservation, and SD-WAN path compliance. [KB-MTG-2006, chunk 5]                                            │
│                                                                                                                 │
│ SD-WAN QoS / SLA to apply                                                                                       │
│                                                                                                                 │
│  • SLA class: NetCore SD-WAN’s current voice/video QoS article classifies NetCore Meet media matched on UDP     │
│    destination port 9000 into the Video SLA class and q-video-priority queue. [KB-NET-1010, Table 1, section    │
│    'Overview']                                                                                                  │
│  • Video SLA thresholds: Video paths should meet latency <= 200 ms, jitter <= 40 ms, and loss <= 2%.            │
│    [KB-NET-1001, Table 1, p.1]                                                                                  │
│  • Meet media classification in the SD-WAN baseline: KB-NET-1010 lists NetCore Meet media | UDP destination     │
│    port 9000 | AF41 34 | Video | q-video-priority. [KB-NET-1010, Table 1, section 'Overview']                   │
│  • Important DSCP note: The Meet troubleshooting guide gives a Meet-specific QoS recommendation to mark         │
│    real-time media with DSCP 46 and meeting signalling with DSCP 24, and to preserve markings across controlled │
│    access, distribution, WAN, and internet-edge policies. [KB-MTG-2006, chunk 5]                                │
│    The provided current sources do not reconcile the DSCP difference between the SD-WAN baseline’s AF41 34      │
│    entry and the Meet guide’s DSCP 46 media recommendation. [KB-NET-1010, Table 1, section 'Overview'];         │
│    [KB-MTG-2006, chunk 5]                                                                                       │
│                                                                                                                 │
│ Meet troubleshooting guide recommendations for MTG-MED-4102                                                     │
│                                                                                                                 │
│  • Packet loss trigger: MTG-MED-4102 is associated with packet loss 3% or higher in in-meeting statistics.      │
│    [KB-MTG-2006, Table 1, section 'Probable Causes']                                                            │
│  • Likely causes: Congestion, lossy Wi‑Fi, overloaded access circuit, or impaired WAN path. [KB-MTG-2006, Table │
│    1, section 'Probable Causes']                                                                                │
│  • Immediate actions: Move to wired access if possible, reduce active HD video streams, check site interface    │
│    utilization, and validate QoS handling. [KB-MTG-2006, Table 1, section 'Probable Causes']                    │
│  • Expected recovery: Packet loss should drop below 3% and MTG-MED-4102 should clear. [KB-MTG-2006, Table 1,    │
│    section 'Probable Causes']                                                                                   │
│  • Preferred media path: Stabilize UDP 9000; TCP 5004 is supported fallback but should not be the preferred     │
│    steady-state path when UDP 9000 can be permitted. [KB-MTG-2006, chunk 5]                                     │
│  • Firewall policy intent: Allow tcp/443 signalling, udp/9000 media, and tcp/5004 media fallback. [KB-MTG-2006, │
│    chunk 5]                                                                                                     │
│  • If UDP 9000 is blocked: Update firewall policy to a

Executed plan                                                                                                      
┏━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━┓
┃ Ver ┃ Step ┃ Goal                                 ┃ Tool                  ┃ Status ┃ Relevant ┃ Best ┃ Time (s) ┃
┡━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━┩
│ v1  │ 1    │ Find the NetCore Meet                │ search_knowledge_base │ done   │ 4        │ 0.95 │ 3.2      │
│     │      │ troubleshooting guidance for         │                       │        │          │      │          │
│     │      │ MTG-MED-4                            │                       │        │          │      │          │
│ v1  │ 2    │ Find the recommended NetCore SD-WAN  │ search_knowledge_base │ done   │ 6        │ 0.95 │ 3.0      │
│     │      │ QoS settings and SLA cla             │                       │        │          │      │          │
└─────┴──────┴──────────────────────────────────────┴───────────────────────┴────────┴──────────┴──────┴──────────┘

Sources                                                                                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ KB ID       ┃ Relevance ┃ Access   ┃ Status  ┃ Location                                 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ KB-MTG-2006 │ 0.95      │ internal │ current │ Table 1, section 'Probable Causes'       │
│ KB-MTG-2006 │ 0.95      │ internal │ current │ chunk 5                                  │
│ KB-NET-1010 │ 0.95      │ internal │ current │ chunk 1                                  │
│ KB-NET-1002 │ 0.90      │ internal │ current │ Table 1, p.1                             │
│ KB-MTG-2006 │ 0.85      │ internal │ current │ Code block 1, section 'Diagnostic Steps' │
│ KB-NET-1010 │ 0.85      │ internal │ current │ Table 1, section 'Overview'              │
│ KB-NET-1001 │ 0.85      │ internal │ current │ Table 1, p.1                             │
│ KB-NET-1010 │ 0.80      │ internal │ current │ Code block 3, section 'Verification'     │
│ KB-NET-1002 │ 0.75      │ internal │ current │ p.1                                      │
│ KB-MTG-2006 │ 0.70      │ internal │ current │ chunk 6                                  │
└─────────────┴───────────┴──────────┴─────────┴──────────────────────────────────────────┘

Judge (RAGAS, gpt-6.1-sol)                        
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric            ┃ Score ┃ Threshold ┃ Result ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ faithfulness      │ 1.000 │ 0.70      │ pass   │
│ groundedness      │ 1.000 │ 0.70      │ pass   │
│ context_precision │ 1.000 │ 0.60      │ pass   │
└───────────────────┴───────┴───────────┴────────┘

In [3]:
with override_settings(timeout_seconds=300):
    show_result(ask("Our NetCore Meet calls show MTG-MED-4102 packet loss at branch sites. Which SD-WAN QoS settings "
                    "and SLA class should we apply, and what does the Meet troubleshooting guide recommend?",
                    "bob", verbose=False, mode="plan"), show_trace=True)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Our NetCore Meet calls show MTG-MED-4102 packet loss at branch sites. Which SD-WAN QoS settings and SLA class   │
│ should we apply, and what does the Meet troubleshooting guide recommend?                                        │
│ user=bob (role employee)  mode=plan  status=cache_hit  routes=-  rewrites/re-plans=0  time=3.6s                 │
│ served from cache (cached question: Our NetCore Meet calls show MTG-MED-4102 packet loss at branch sites. Which │
│ SD-WAN QoS settings and SLA class should we apply, and what does the Meet troubleshooting guide recommend?)     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ For MTG-MED-4102 at branch sites, treat this as a media-path quality issue and prioritize UDP/9000 stability,   │
│ QoS preservation, and SD-WAN path compliance. [KB-MTG-2006, chunk 5]                                            │
│                                                                                                                 │
│ SD-WAN QoS / SLA to apply                                                                                       │
│                                                                                                                 │
│  • SLA class: NetCore SD-WAN’s current voice/video QoS article classifies NetCore Meet media matched on UDP     │
│    destination port 9000 into the Video SLA class and q-video-priority queue. [KB-NET-1010, Table 1, section    │
│    'Overview']                                                                                                  │
│  • Video SLA thresholds: Video paths should meet latency <= 200 ms, jitter <= 40 ms, and loss <= 2%.            │
│    [KB-NET-1001, Table 1, p.1]                                                                                  │
│  • Meet media classification in the SD-WAN baseline: KB-NET-1010 lists NetCore Meet media | UDP destination     │
│    port 9000 | AF41 34 | Video | q-video-priority. [KB-NET-1010, Table 1, section 'Overview']                   │
│  • Important DSCP note: The Meet troubleshooting guide gives a Meet-specific QoS recommendation to mark         │
│    real-time media with DSCP 46 and meeting signalling with DSCP 24, and to preserve markings across controlled │
│    access, distribution, WAN, and internet-edge policies. [KB-MTG-2006, chunk 5]                                │
│    The provided current sources do not reconcile the DSCP difference between the SD-WAN baseline’s AF41 34      │
│    entry and the Meet guide’s DSCP 46 media recommendation. [KB-NET-1010, Table 1, section 'Overview'];         │
│    [KB-MTG-2006, chunk 5]                                                                                       │
│                                                                                                                 │
│ Meet troubleshooting guide recommendations for MTG-MED-4102                                                     │
│                                                                                                                 │
│  • Packet loss trigger: MTG-MED-4102 is associated with packet loss 3% or higher in in-meeting statistics.      │
│    [KB-MTG-2006, Table 1, section 'Probable Causes']                                                            │
│  • Likely causes: Congestion, lossy Wi‑Fi, overloaded access circuit, or impaired WAN path. [KB-MTG-2006, Table │
│    1, section 'Probable Causes']                                                                                │
│  • Immediate actions: Move to wired access if possible, reduce active HD video streams, check site interface    │
│    utilization, and validate QoS handling. [KB-MTG-2006, Table 1, section 'Probable Causes']                    │
│  • Expected recovery: Packet loss should drop below 3% and MTG-MED-4102 should clear. [KB-MTG-2006, Table 1,    │
│    section 'Probable Causes']                                                                                   │
│  • Preferred media path: Stabilize UDP 9000; TCP 5004 is supported fallback but should not be the preferred     │
│    steady-state path when UDP 9000 can be permitted. [KB-MTG-2006, chunk 5]                                     │
│  • Firewall policy intent: Allow tcp/443 signalling, udp/9000 media, and tcp/5004 media fallback. [KB-MTG-2006, │
│    chunk 5]                                                                                                     │
│  • If UDP 9000 is blocked: Update firewall policy to a

Sources                                                   
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ KB ID       ┃ Relevance ┃ Access   ┃ Status ┃ Location ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ KB-MTG-2006 │ -         │ internal │        │ cached   │
│ KB-NET-1001 │ -         │ internal │        │ cached   │
│ KB-NET-1002 │ -         │ internal │        │ cached   │
│ KB-NET-1010 │ -         │ internal │        │ cached   │
└─────────────┴───────────┴──────────┴────────┴──────────┘

Trace                                                                                                              
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   t (s) ┃ Step                 ┃ Decision               ┃ Detail                                                ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    0.00 │ Request              │ bob                    │ role=employee mode=plan q='Our NetCore Meet calls     │
│         │                      │                        │ show MTG-MED-4102 packet loss at branch sites. Which  │
│         │                      │                        │ SD-WAN QoS settings and SLA class should we apply,    │
│         │                      │                        │ and what does the Meet troubleshooting guide r        │
│    1.89 │ Input guardrails     │ allowed                │ Asks about NetCore Meet error, SD-WAN QoS, and SLA    │
│         │                      │                        │ class, which is within allowed NetCore and enterprise │
│         │                      │                        │ networking scope.                                     │
│    2.27 │ Semantic cache       │ hit                    │ similarity=1.0 level=internal scope=internal          │
│         │                      │                        │ cached_q='Our NetCore Meet calls show MTG-MED-4102    │
│         │                      │                        │ packet loss at branch sites. Which SD-WAN QoS         │
│         │                      │                        │ settings and SLA class should we apply, and what does │
│         │                      │                        │ the                                                   │
│    3.56 │ Cache hit checks     │ pass                   │ RBAC + output guardrails                              │
│    3.56 │ Response             │ cache_hit              │ 3667 chars in 3.6s                                    │
└─────────┴──────────────────────┴────────────────────────┴───────────────────────────────────────────────────────┘